# 03 — Simulator vs Real QPU

Same Bell circuit as notebook 02 — but this time we compare **Aer**, a classical simulator
(fast, free, noiseless), against a **real IBM Quantum backend** (real superconducting qubits,
subject to decoherence and gate error).

> **Running this notebook** — from the repo root:
> ```bash
> uv run --with jupyterlab --with qiskit==1.2.0 --with matplotlib --with pylatexenc --with httpx \
>   jupyter lab notebooks/03-simulator-vs-qpu.ipynb
> ```
> Start the FastAPI backend in another terminal:
> ```bash
> cd backend && uv run uvicorn main:app --reload
> ```
> The real-QPU step only runs if the backend has an `IBMQ_API_TOKEN` set (see `.env.example`).
> Without one, everything below still works entirely on the simulator.

## Step 0 — Install dependencies

In [ ]:
%pip install -q qiskit==1.2.0 matplotlib pylatexenc httpx
print("✅ Qiskit + matplotlib + pylatexenc + httpx ready")

## Step 0b — Load the workshop helpers (`notebooks/lib/`)

In [ ]:
import sys
from pathlib import Path

_notebooks_dir = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd() / "notebooks"
if str(_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(_notebooks_dir))

from lib.visuals import draw_circuit, plot_counts, run_and_plot
from lib.backend import run_via_backend, list_backends

## Step 1 — What backends are available?

In [ ]:
backends = list_backends()
print("Available backends:", backends)
has_real_qpu = any(b != "aer_simulator" for b in backends)

If only `aer_simulator` is listed, the backend server has no `IBMQ_API_TOKEN`
configured — that's expected for a local dev setup, and everything below still runs.

## Step 2 — The circuit under test

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

bell_code = """from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
"""

draw_circuit(qc)

## Step 3 — Run on the simulator (baseline)

In [ ]:
sim_counts = run_via_backend(bell_code, backend="aer_simulator")

if sim_counts is None:
    print("Backend not reachable at http://localhost:8000 — running locally instead.")
    sim_counts = run_and_plot(qc, title="Bell state — aer_simulator (local fallback)")
else:
    plot_counts(sim_counts, title="Bell state — aer_simulator")

## Step 4 — Run on real hardware (if available)

In [ ]:
if has_real_qpu:
    real_backend = next(b for b in backends if b != "aer_simulator")
    print(f"Running on real hardware: {real_backend} (real queues can take a while)")
    qpu_counts = run_via_backend(bell_code, backend=real_backend, timeout=120.0)
    if qpu_counts:
        plot_counts(qpu_counts, title=f"Bell state — {real_backend}")
    else:
        print("Real-hardware run did not complete in time, or failed — the queue can be long.")
else:
    print(
        "No IBMQ_API_TOKEN configured — skipping the real-hardware run. On real hardware "
        "you'd expect the same 00/11 peaks as the simulator, but with some 01/10 counts "
        "leaking in from decoherence and gate error (noise the simulator doesn't have)."
    )

## Step 5 — Read it

The **same Qiskit code** targets either backend — only the `backend` name passed to
`/api/circuit/run` changes. That's the whole point: simulators are for fast iteration, and real
QPUs are where quantum advantage would actually have to hold up.

## What's next?

Continue to **notebook 04** to see *why* quantum can be faster at all — the classical-vs-quantum
parity check and the oracle idea that Grover generalizes.

---
*Quantum Experience Lab — IBM Bob + Qiskit workshop*